# v4.3 — Black-Scholes Backtest (Long History)

| | |
|---|---|
| **Version** | v4.3 |
| **Key change** | SGX live at 9:25 IST (BS uses daily close fallback) |
| **Data source** | Black-Scholes simulation using yfinance daily data |
| **Period** | Configurable `BS_START_DATE` (default 2012-01-01) to 2026-04-15 |
| **Lookahead** | None |

⚠️ **BS limitations apply** — see printed warning at start of Cell 5.
OHLC-proxy SL/TP and no volatility smile. Use for directional confirmation only.


In [6]:
VERSION     = 'v4.3_bs'
KEY_CHANGE  = 'SGX live at 9:25 IST (BS uses daily close fallback)'
DATA_SRC    = 'black_scholes'

BS_START_DATE = '2019-01-01'   # change to '2019-01-01' for shorter run
BS_END_DATE   = '2026-04-15'

SL_PCT    = 0.15
TP_PCT    = 0.40
BASE_LOTS = 2
MAX_LOTS  = 10
DTE0_LOTS = 5
LOT_SIZE  = 75
BEAR_N    = 10
BASE_RATE = 54.5
ENTRY_TIME      = '09:25'
EXIT_TIME       = '11:15'
RISK_FREE       = 0.065   # Indian T-bill proxy
INITIAL_CAPITAL = 3000.0  # fixed starting capital for all versions (Rs)

print(f"Backtest period: {BS_START_DATE} to {BS_END_DATE}")
print(f"Breakeven win rate: {SL_PCT/(SL_PCT+TP_PCT)*100:.1f}%")


Backtest period: 2019-01-01 to 2026-04-15
Breakeven win rate: 27.3%


In [7]:
import sys, os
from pathlib import Path

HERE    = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
# HERE.parent.parent == gap_trading/ for both v2/backtest and v4/v4.x layouts
V4_DIR  = HERE.parent.parent / 'v4'
sys.path.insert(0, str(V4_DIR))

from _engine   import (compute_metrics, save_results, sig_map,
                        combo_fires, load_reliable_signals, print_summary)
from _bs_engine import (bs_put_price, simulate_bs_exit, vix_to_sigma,
                        dte_to_T, print_bs_limitations)

import pandas as pd
import numpy  as np
import yfinance as yf
import warnings
from datetime import date, timedelta
from pathlib  import Path
warnings.filterwarnings('ignore')


In [8]:
# ── Paths ──────────────────────────────────────────────────────────────────────
SIGNALS_CSV = HERE.parent.parent / 'v2' / 'v2_reliable_signals.csv'
if not SIGNALS_CSV.exists():
    raise FileNotFoundError(f"Signals CSV not found: {SIGNALS_CSV}")

# ── Global market data ─────────────────────────────────────────────────────────
print("Fetching global daily data ...", end=' ', flush=True)
TICKERS = {'SP500': '^GSPC', 'SGX': 'NKD=F', 'DAX': '^GDAXI',
           'VIX': '^VIX', 'NIFTY': '^NSEI', 'INDIAVIX': '^INDIAVIX'}

raw = {}
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=BS_START_DATE, end=BS_END_DATE,
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        df.index = df.index.date
        raw[name] = df[['Open','High','Low','Close']]
    except Exception as e:
        print(f"  [warn] {name}: {e}")
        raw[name] = pd.DataFrame()
print("done.")

nifty   = raw.get('NIFTY',    pd.DataFrame())
sp500   = raw.get('SP500',    pd.DataFrame())
sgx     = raw.get('SGX',      pd.DataFrame())
dax_df  = raw.get('DAX',      pd.DataFrame())
vix_df  = raw.get('VIX',      pd.DataFrame())
ivix_df = raw.get('INDIAVIX', pd.DataFrame())

# ── Signal combos ──────────────────────────────────────────────────────────────
bear_combos = load_reliable_signals(SIGNALS_CSV, bear_n=BEAR_N, base_rate=BASE_RATE)
print(f"Loaded {len(bear_combos)} bearish combos.")

# ── NSE weekly expiry: Thursday before Sep 2 2025, Tuesday from Sep 2 2025 ───
from datetime import date as _date
EXPIRY_CHANGE = _date(2025, 9, 2)
def next_expiry(d):
    wd_target = 1 if d >= EXPIRY_CHANGE else 3   # Tue=1, Thu=3
    days = (wd_target - d.weekday()) % 7
    return d + timedelta(days=days)

# ── NSE holidays (rough annual calendar) ──────────────────────────────────────
NSE_HOLIDAYS = set()   # kept minimal — BS backtest spans years, exact holidays skipped



# ── Build signal days + BS simulation ─────────────────────────────────────────
from _engine import compute_charges

trade_log = []
capital   = None
peak      = None
trade_num = 0
SKIP_MON  = True   # keep Monday skip consistent with v2 baseline

all_dates = sorted([d for d in nifty.index
                    if isinstance(d, _date)
                    and _date.fromisoformat(BS_START_DATE) <= d
                    and d <= _date.fromisoformat(BS_END_DATE)])

for d in all_dates:
    if SKIP_MON and d.weekday() == 0: continue
    if d in NSE_HOLIDAYS: continue

    prev_rows = [x for x in nifty.index if x < d]
    if len(prev_rows) < 2: continue
    prev, prev2 = prev_rows[-1], prev_rows[-2]

    def _r(df, d1, d2):
        try:
            return float((df.loc[d1,'Close'] - df.loc[d2,'Close']) / df.loc[d2,'Close'])
        except Exception: return None

    def _c(df, d1):
        try: return float(df.loc[d1,'Close'])
        except Exception: return None

    sp500_ret = _r(sp500, prev, prev2)
    sgx_ret   = _r(sgx,   prev, prev2)
    dax_ret   = _r(dax_df,prev, prev2)
    vix_ret   = _r(vix_df, prev, prev2)
    pind_ret  = _r(nifty,  prev, prev2)
    vix_india = _c(ivix_df, prev)

    # ── Gap: use NIFTY open on trade day ──────────────────────────────────
    try:
        nifty_open  = float(nifty.loc[d, 'Open'])
        nifty_prev_close = float(nifty.loc[prev, 'Close'])
        if nifty_prev_close <= 0: continue
    except Exception: continue

    gap = (nifty_open - nifty_prev_close) / nifty_prev_close

    sigs  = sig_map(gap, pind_ret, sp500_ret, sgx_ret, dax_ret, vix_ret)
    fired = [c for c in bear_combos if combo_fires(c, sigs)]
    if not fired: continue

    # ── Strike + expiry + BS pricing ──────────────────────────────────────
    STRIKE_STEP = 50
    atm    = round(nifty_open / STRIKE_STEP) * STRIKE_STEP
    strike = atm - STRIKE_STEP
    exp    = next_expiry(d)
    dte    = (exp - d).days

    sigma     = vix_to_sigma(vix_india)
    T_en, T_ex = dte_to_T(dte)
    entry_price = bs_put_price(nifty_open, strike, T_en, RISK_FREE, sigma)
    if entry_price <= 0.5: continue

    # ── NIFTY daily OHLC for exit simulation ──────────────────────────────
    try:
        S_high  = float(nifty.loc[d, 'High'])
        S_low   = float(nifty.loc[d, 'Low'])
        S_close = float(nifty.loc[d, 'Close'])
    except Exception: continue

    exit_price, exit_reason = simulate_bs_exit(
        entry_price, nifty_open, S_high, S_low, S_close,
        strike, T_en, T_ex, SL_PCT, TP_PCT, RISK_FREE, sigma, dte
    )

    # ── Capital + lot sizing ───────────────────────────────────────────────
    cost_per_lot = entry_price * LOT_SIZE
    if capital is None:
        capital = float(INITIAL_CAPITAL) if INITIAL_CAPITAL is not None else cost_per_lot * BASE_LOTS
        peak    = capital
    refill = 0.0
    if capital < cost_per_lot:
        refill  = cost_per_lot * BASE_LOTS - capital
        capital += refill
        peak = max(peak, capital)

    lots = min(max(int(capital // cost_per_lot), 1), MAX_LOTS)
    if dte == 0: lots = min(lots, DTE0_LOTS)

    charges = compute_charges(entry_price, exit_price, lots, LOT_SIZE)
    pnl_rs  = round((exit_price - entry_price) * lots * LOT_SIZE - charges, 4)

    cap_before = round(capital, 4)
    capital    = round(capital + pnl_rs, 4)
    peak       = max(peak, capital)
    dd         = round((peak - capital) / peak * 100, 4) if peak > 0 else 0.0

    trade_num += 1
    trade_log.append(dict(
        trade_num=trade_num, date=str(d), strike=strike, dte=dte,
        entry=round(entry_price,4), lots=lots,
        sl_price=round(entry_price*(1-SL_PCT),4),
        tp_price=round(entry_price*(1+TP_PCT),4),
        exit_price=round(exit_price,4), exit_reason=exit_reason,
        exit_time='BS-approx', pnl_pts=round(exit_price-entry_price,4),
        pnl_rs=pnl_rs, charges_rs=round(charges,4),
        capital_before=cap_before, capital_after=capital,
        drawdown_pct=dd, combo=fired[0],
        refill_rs=round(refill, 4),
        dte0_warning=(dte == 0),
    ))

df_log = pd.DataFrame(trade_log)
print(f"Simulation complete: {len(df_log)} trades, period {BS_START_DATE}–{BS_END_DATE}")
df_log.head()


Fetching global daily data ... done.
Loaded 10 bearish combos.
Simulation complete: 343 trades, period 2019-01-01–2026-04-15


,trade_num,date,strike,dte,entry,lots,sl_price,tp_price,exit_price,exit_reason,exit_time,pnl_pts,pnl_rs,charges_rs,capital_before,capital_after,drawdown_pct,combo,refill_rs,dte0_warning
0,1,2019-01-09,10800,1,11.8799,3,10.0979,16.6319,10.0979,Stop Loss,BS-approx,-1.7820,-452.7530,51.7987,3000.0000,2547.2470,15.0918,Gap Up + SGX UP + DAX UP + VIX Falling,0.000,False
1,2,2019-01-30,10650,1,19.0971,1,16.2325,26.7360,16.2325,Stop Loss,BS-approx,-2.8646,-264.5098,49.6641,2547.2470,2282.7372,23.9088,Gap Up + Prev India DOWN + SGX UP + DAX UP,0.000,False
2,3,2019-02-01,10800,6,66.4225,2,56.4591,92.9914,56.4591,Stop Loss,BS-approx,-9.9634,-1558.8454,64.3412,9963.3692,8404.5238,15.6458,Gap Up + SGX UP + VIX Falling,7680.632,False
3,4,2019-02-06,10900,1,11.6434,9,9.8969,16.3008,9.8969,Stop Loss,BS-approx,-1.7465,-1239.6264,60.7213,8404.5238,7164.8974,28.0876,Gap Up + SGX UP + DAX UP + VIX Falling,0.000,False
4,5,2019-02-15,10750,6,67.3452,1,57.2434,94.2833,94.2833,Target Hit,BS-approx,26.9381,1960.9885,59.3665,7164.8974,9125.8859,8.4056,Gap Up + Prev India DOWN + US UP + SGX UP,0.000,False


In [9]:
params = dict(
    SL_PCT=SL_PCT, TP_PCT=TP_PCT, BASE_LOTS=BASE_LOTS,
    MAX_LOTS=MAX_LOTS, DTE0_LOTS=DTE0_LOTS, LOT_SIZE=LOT_SIZE,
    BEAR_N=BEAR_N, ENTRY_TIME=ENTRY_TIME, EXIT_TIME=EXIT_TIME,
    INITIAL_CAPITAL=INITIAL_CAPITAL,
)
metrics = compute_metrics(df_log, params)


In [10]:
print_bs_limitations()
print_summary(df_log, metrics, params)

dte0_trades = df_log.get('dte0_warning', pd.Series(dtype=bool))
if hasattr(dte0_trades, 'sum') and dte0_trades.sum() > 0:
    print(f"  ⚠ DTE=0 trades: {dte0_trades.sum()} — BS unreliable for these")

version_meta = dict(version=VERSION, key_change=KEY_CHANGE, data_source=DATA_SRC)
out = save_results(metrics, params, version_meta, HERE / 'results')
print(f"Run ID: {out.stem}")



╔══════════════════════════════════════════════════════════════════════╗
║  BLACK-SCHOLES BACKTEST — KNOWN LIMITATIONS                         ║
║                                                                      ║
║  1. SL/TP TIMING unknown: OHLC proxy cannot detect intraday order.  ║
║     Results may overstate wins (TP) or losses (SL) on volatile days.║
║  2. Volatility smile ignored: BS underprices deep-OTM options.      ║
║  3. DTE=0 trades are flagged — extreme gamma makes BS unreliable.   ║
║  4. Pre-2021 data is OUT-OF-SAMPLE for v2 signal combos.            ║
║     Combos were statistically selected on 2021–2026 data.           ║
║  Use these results for directional confirmation only.                ║
╚══════════════════════════════════════════════════════════════════════╝

  BACKTEST RESULTS  ·  2019-01-09 to 2026-04-01
  Trades          : 343
  TP hit          : 18.7%  (64 trades)
  SL hit          : 80.2%  (275 trades)
  Time exit       : 1.2%  (4 trades)
  Profitable  